# Interpretable Predictions for Steam Game Recommendations

**Author:** Michael Theophanopoulos  
**Purpose:** Binary classifier with counterfactual explanations  
**Last Updated:** 2025-10-29

---

## Notebook Structure

1. Data Loading & Exploration
2. Feature Engineering
3. Model Training
4. Prediction Function (with Counterfactuals)
5. Model Evaluation

In [68]:
# Core libraries
import pandas as pd
import numpy as np
from pathlib import Path
import logging
import sys
from datetime import datetime

# ML libraries
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score

def setup_logger(name: str = 'steam_predictions', level: int = logging.INFO) -> logging.Logger:
    """
    Configure enterprise-level logging for the notebook.
    
    Best practices implemented:
    - Structured log format with timestamp, level, and message
    - Both console and file output
    - Proper log rotation prevention (overwrites for notebooks)
    - Clear, hierarchical logger naming
    
    Parameters
    ----------
    name : str
        Logger name (hierarchical, e.g., 'steam_predictions.features')
    level : int
        Logging level (DEBUG, INFO, WARNING, ERROR, CRITICAL)
        
    Returns
    -------
    logging.Logger
        Configured logger instance
    """
    # Create logger
    logger = logging.getLogger(name)
    logger.setLevel(level)
    
    # Prevent duplicate handlers in notebook environment
    if logger.handlers:
        logger.handlers.clear()
    
    # Create formatters
    console_formatter = logging.Formatter(
        fmt='%(levelname)-8s | %(message)s',
        datefmt='%Y-%m-%d %H:%M:%S'
    )
    
    file_formatter = logging.Formatter(
        fmt='%(asctime)s | %(name)-25s | %(levelname)-8s | %(message)s',
        datefmt='%Y-%m-%d %H:%M:%S'
    )
    
    # Console handler (for notebook output)
    console_handler = logging.StreamHandler(sys.stdout)
    console_handler.setLevel(level)
    console_handler.setFormatter(console_formatter)
    logger.addHandler(console_handler)
    
    # File handler (for audit trail)
    log_file = Path('logs') / f'steam_predictions_{datetime.now():%Y%m%d}.log'
    log_file.parent.mkdir(exist_ok=True)
    
    file_handler = logging.FileHandler(log_file, mode='a', encoding='utf-8')
    file_handler.setLevel(logging.DEBUG)  # Capture everything in file
    file_handler.setFormatter(file_formatter)
    logger.addHandler(file_handler)
    
    # Prevent propagation to root logger
    logger.propagate = False
    
    return logger

# Initialize main logger
logger = setup_logger('steam_predictions', level=logging.INFO)

logger.info("="*60)
logger.info("Steam Game Recommendation Prediction System")
logger.info(f"Notebook initialized at {datetime.now():%Y-%m-%d %H:%M:%S}")
logger.info("="*60)

INFO     | ============================================================
INFO     | Steam Game Recommendation Prediction System
INFO     | Notebook initialized at 2025-10-30 13:07:10
INFO     | ============================================================


---

## 0. Data Scraping

Scape the data from https://store.steampowered.com/appreviews/ and store them in a csv file.

In [48]:
import requests
import pandas as pd
import time
from pathlib import Path
from typing import List, Dict, Optional
import logging

In [77]:
def scrape_steam_reviews(
    app_id: int,
    max_reviews: int = 50000,
    language: str = 'english',
    reviews_per_page: int = 100
) -> List[Dict]:
    """
    Scrape game reviews from Steam API.
    
    Parameters
    ----------
    app_id : int
        Steam application ID (default: 1245620 for Elden Ring)
    max_reviews : int
        Maximum number of reviews to collect
    language : str
        Review language filter
    reviews_per_page : int
        Number of reviews per API request (max 100)
        
    Returns
    -------
    List[Dict]
        List of review dictionaries
    """
    all_reviews = []
    cursor = '*'
    start_time = time.time()
    
    logging.info(f"Starting fetching reviews for app_id={app_id}...")
    
    while len(all_reviews) < max_reviews:
        url = f'https://store.steampowered.com/appreviews/{app_id}'
        params = {
            'json': 1,
            'language': language,
            'cursor': cursor,
            'num_per_page': reviews_per_page,
            'filter': 'recent'
        }
        
        try:
            response = requests.get(url, params=params, timeout=10)
            response.raise_for_status()
            data = response.json()
            
            if not data.get('reviews'):
                logging.warning("No more reviews available")
                break
            
            all_reviews.extend(data['reviews'])
            cursor = data.get('cursor')
            
            if not cursor:
                logging.warning("No cursor returned, ending pagination")
                break
            
            # Rate limiting: 0.3s between requests
            time.sleep(0.3)
            
        except requests.exceptions.RequestException as e:
            logging.error(f"Request failed: {e}")
            time.sleep(5)
    
    final_reviews = all_reviews[:max_reviews]
    elapsed_time = time.time() - start_time
    
    logging.info(f"Fetched {len(final_reviews)} reviews in {elapsed_time:.2f} seconds")
    
    return final_reviews

In [78]:
def fetch_game_details(app_id: int) -> Dict:
    """
    Fetch game metadata from Steam Store API.
    
    Parameters
    ----------
    app_id : int
        Steam application ID
        
    Returns
    -------
    Dict
        Game metadata including price, genres, description, etc.
    """
    url = f'https://store.steampowered.com/api/appdetails'
    params = {'appids': app_id}
    
    try:
        response = requests.get(url, params=params, timeout=10)
        response.raise_for_status()
        data = response.json()
        
        if str(app_id) in data and data[str(app_id)]['success']:
            game_data = data[str(app_id)]['data']
            logging.info(f"Fetched game details for '{game_data.get('name', 'Unknown')}'")
            return game_data
        else:
            logging.warning(f"Failed to fetch game details for app_id={app_id}")
            return {}
            
    except requests.exceptions.RequestException as e:
        logging.error(f"Request failed: {e}")
        return {}


def process_reviews(reviews: List[Dict], game_data: Dict = None) -> pd.DataFrame:
    """
    Convert raw review data to structured DataFrame.
    
    Parameters
    ----------
    reviews : List[Dict]
        Raw review data from Steam API
    game_data : Dict, optional
        Game metadata from appdetails API (price, genre, description, etc.)
        
    Returns
    -------
    pd.DataFrame
        Structured dataset with selected features for binary classification
    """
    data = []
    
    for review in reviews:
        author = review.get('author', {})
        
        record = {
            # Target variable
            'recommended': review.get('voted_up', False),
            
            # User attributes (descriptive features)
            'user_id': author.get('steamid', ''),
            'user_games_owned': author.get('num_games_owned', 0),
            'user_num_reviews': author.get('num_reviews', 0),
            
            # User-Game interaction (numeric features)
            'playtime_at_review_hours': author.get('playtime_at_review', 0) / 60,
            'playtime_total_hours': author.get('playtime_forever', 0) / 60,
            'playtime_recent_hours': author.get('playtime_last_two_weeks', 0) / 60,
            
            # Review metadata (categorical features)
            'received_free': review.get('received_for_free', False),
            'steam_purchase': review.get('steam_purchase', True),
            'written_early_access': review.get('written_during_early_access', False),
            
            # Review engagement (numeric features)
            'votes_helpful': review.get('votes_up', 0),
            'votes_funny': review.get('votes_funny', 0),
            'weighted_vote_score': review.get('weighted_vote_score', 0.0),
            'comment_count': review.get('comment_count', 0),
            
            # Temporal features
            'timestamp_created': review.get('timestamp_created', 0),
            'timestamp_updated': review.get('timestamp_updated', 0),
            
            # Text features (free text)
            'review_text': review.get('review', ''),
            'review_length': len(review.get('review', '')),
            
            # Unique identifiers
            'review_id': review.get('recommendationid', '')
        }
        
        # Add game features if game_data provided
        if game_data:
            record.update({
                'game_id': game_data.get('steam_appid', ''),
                'game_name': game_data.get('name', ''),
                'game_price': game_data.get('price_overview', {}).get('final', 0) / 100 if game_data.get('price_overview') else 0,
                'game_is_free': game_data.get('is_free', False),
                'game_genre': ','.join([g['description'] for g in game_data.get('genres', [])]),
                'game_categories': ','.join([c['description'] for c in game_data.get('categories', [])]),
                'game_developer': ','.join(game_data.get('developers', [])),
                'game_publisher': ','.join(game_data.get('publishers', [])),
                'game_description': game_data.get('short_description', ''),
                'game_required_age': game_data.get('required_age', 0),
            })
        
        data.append(record)
    
    df = pd.DataFrame(data)
    
    # Feature engineering
    df['playtime_ratio'] = df['playtime_at_review_hours'] / (df['playtime_total_hours'] + 1)
    df['review_engagement_score'] = df['votes_helpful'] + df['votes_funny'] * 0.5
    df['experienced_gamer'] = df['user_games_owned'] > 50
    df['active_reviewer'] = df['user_num_reviews'] > 5
    
    feature_count = len(df.columns)
    logging.info(f"Processed {len(df)} reviews into DataFrame with {feature_count} features")
    
    return df

In [ ]:
# Configuration
APP_ID = 578080  # PUBG
MAX_REVIEWS = 50000
OUTPUT_FILE = 'steam_reviews.csv'

# Fetch game metadata first
logging.info(f"Fetching game metadata for app_id={APP_ID}...")
game_data = fetch_game_details(APP_ID)

# Collect reviews
logging.info(f"Starting to scrape reviews for app_id={APP_ID}...")
reviews = scrape_steam_reviews(app_id=APP_ID, max_reviews=MAX_REVIEWS)

In [ ]:
# Process reviews with game data and save to CSV
df = process_reviews(reviews, game_data=game_data)
df.to_csv(OUTPUT_FILE, index=False)
logging.info(f"Dataset saved to {OUTPUT_FILE}")

In [ ]:
# Data summary
print(f"Dataset shape: {df.shape}")
print(f"Positive reviews: {df['recommended'].sum()} ({df['recommended'].mean()*100:.1f}%)")

# Show game info if available
if 'game_name' in df.columns:
    print(f"\nGame: {df['game_name'].iloc[0]}")
    print(f"Price: ${df['game_price'].iloc[0]:.2f}")
    print(f"Genre: {df['game_genre'].iloc[0]}")
    print(f"Description: {df['game_description'].iloc[0][:100]}...")

print(f"\nFirst few rows:")
df.head()

---

## 1. Data Loading & Exploration

Load the preprocessed Steam reviews dataset and perform exploratory analysis.

In [80]:
# File paths
DATA_FILE = Path('steam_reviews.csv')
# Validate file exists
if not DATA_FILE.exists():
    logger.error(f"Data file not found: {DATA_FILE}")
    raise FileNotFoundError(
        f"Data file not found: {DATA_FILE}. "
        "Please run data collection first."
    )

# Load data
logger.info(f"Loading data from {DATA_FILE}")
df = pd.read_csv(DATA_FILE)
logger.info(f"Loaded {len(df):,} reviews with {df.shape[1]} columns")

# Display shape
print(f"\nDataset shape: {df.shape}")
df.head()

INFO     | Loading data from steam_reviews.csv
INFO     | Loaded 50,000 reviews with 33 columns

Dataset shape: (50000, 33)


,recommended,user_id,user_games_owned,user_num_reviews,playtime_at_review_hours,playtime_total_hours,playtime_recent_hours,received_free,steam_purchase,written_early_access,...,game_genre,game_categories,game_developer,game_publisher,game_description,game_required_age,playtime_ratio,review_engagement_score,experienced_gamer,active_reviewer
0,True,76561197990088609,0,11,80.316667,80.316667,0.000000,False,True,False,...,"Action,Adventure,Massively Multiplayer,Free To...","Multi-player,PvP,Online PvP,Stats,Remote Play ...",PUBG Corporation,"KRAFTON, Inc.","PUBG: BATTLEGROUNDS, the high-stakes winner-ta...",0,0.987702,0.0,False,True
1,True,76561198347370385,78,2,351.566667,352.650000,25.266667,False,True,False,...,"Action,Adventure,Massively Multiplayer,Free To...","Multi-player,PvP,Online PvP,Stats,Remote Play ...",PUBG Corporation,"KRAFTON, Inc.","PUBG: BATTLEGROUNDS, the high-stakes winner-ta...",0,0.994109,0.0,True,False
2,False,76561198372979773,280,26,13.900000,13.900000,1.583333,False,True,False,...,"Action,Adventure,Massively Multiplayer,Free To...","Multi-player,PvP,Online PvP,Stats,Remote Play ...",PUBG Corporation,"KRAFTON, Inc.","PUBG: BATTLEGROUNDS, the high-stakes winner-ta...",0,0.932886,4.5,True,True
3,False,76561198027390661,189,13,168.216667,168.216667,0.000000,False,True,False,...,"Action,Adventure,Massively Multiplayer,Free To...","Multi-player,PvP,Online PvP,Stats,Remote Play ...",PUBG Corporation,"KRAFTON, Inc.","PUBG: BATTLEGROUNDS, the high-stakes winner-ta...",0,0.994090,0.0,True,True
4,True,76561198310415102,61,6,218.083333,218.716667,7.516667,False,True,False,...,"Action,Adventure,Massively Multiplayer,Free To...","Multi-player,PvP,Online PvP,Stats,Remote Play ...",PUBG Corporation,"KRAFTON, Inc.","PUBG: BATTLEGROUNDS, the high-stakes winner-ta...",0,0.992566,0.0,True,True


In [70]:
# Dataset overview
print("="*60)
print("DATASET SUMMARY")
print("="*60)
print(f"Total records: {len(df):,}")
print(f"Features: {df.shape[1]}")
print(f"\nTarget distribution:")
print(f"  Positive reviews: {df['recommended'].sum():,} ({df['recommended'].mean()*100:.2f}%)")
print(f"  Negative reviews: {(~df['recommended']).sum():,} ({(~df['recommended']).mean()*100:.2f}%)")
print("\n" + "="*60)

DATASET SUMMARY
Total records: 50,000
Features: 23

Target distribution:
  Positive reviews: 34,833 (69.67%)
  Negative reviews: 15,167 (30.33%)



In [56]:
# Data quality check
print("DATA QUALITY REPORT")
print("="*60)
print("\nMissing values:")
print(df.isnull().sum())
print("\nData types:")
print(df.dtypes)
print("\nBasic statistics:")
df.describe()

DATA QUALITY REPORT

Missing values:
recommended                   0
user_id                       0
user_games_owned              0
user_num_reviews              0
playtime_at_review_hours      0
playtime_total_hours          0
playtime_recent_hours         0
received_free                 0
steam_purchase                0
written_early_access          0
votes_helpful                 0
votes_funny                   0
weighted_vote_score           0
comment_count                 0
timestamp_created             0
timestamp_updated             0
review_text                 188
review_length                 0
review_id                     0
playtime_ratio                0
review_engagement_score       0
experienced_gamer             0
active_reviewer               0
dtype: int64

Data types:
recommended                    bool
user_id                       int64
user_games_owned              int64
user_num_reviews              int64
playtime_at_review_hours    float64
playtime_total_hours 

,user_id,user_games_owned,user_num_reviews,playtime_at_review_hours,playtime_total_hours,playtime_recent_hours,votes_helpful,votes_funny,weighted_vote_score,comment_count,timestamp_created,timestamp_updated,review_length,review_id,playtime_ratio,review_engagement_score
count,5.000000e+04,50000.000000,50000.000000,50000.000000,50000.000000,50000.000000,50000.000000,50000.000000,50000.000000,50000.000000,5.000000e+04,5.000000e+04,50000.000000,5.000000e+04,50000.000000,50000.000000
mean,7.656120e+16,60.827420,9.950820,571.543931,851.852829,2.905052,1.591480,0.328500,0.503188,0.050300,1.665138e+09,1.667795e+09,90.716380,1.258723e+08,0.671733,1.755730
std,4.050778e+08,287.152493,39.024014,1077.466858,1493.023876,11.375891,49.391639,10.005544,0.024541,0.995324,4.113490e+07,4.204586e+07,251.808146,3.278874e+07,0.302666,53.840778
min,7.656120e+16,0.000000,1.000000,0.083333,0.133333,0.000000,0.000000,0.000000,0.132850,0.000000,1.612215e+09,1.612215e+09,0.000000,8.578226e+07,0.000038,0.000000
25%,7.656120e+16,0.000000,1.000000,31.716667,74.829167,0.000000,0.000000,0.000000,0.500000,0.000000,1.631402e+09,1.633258e+09,8.000000,9.919439e+07,0.448158,0.000000
50%,7.656120e+16,0.000000,3.000000,171.291667,293.391667,0.000000,0.000000,0.000000,0.500000,0.000000,1.650211e+09,1.653607e+09,23.000000,1.140148e+08,0.768498,0.000000
75%,7.656120e+16,51.000000,9.000000,646.425000,957.620833,0.000000,0.000000,0.000000,0.500000,0.000000,1.698674e+09,1.702011e+09,73.000000,1.491942e+08,0.940019,0.000000
max,7.656120e+16,33673.000000,6003.000000,30972.666667,43321.100000,297.600000,7228.000000,1519.000000,0.945723,158.000000,1.761706e+09,1.761737e+09,7832.000000,2.078426e+08,1.062945,7987.500000


---

## 2. Feature Engineering

Prepare features from review text, playtime, and other variables for the binary classifier.

In [ ]:
# Feature engineering imports
from textblob import TextBlob

In [ ]:
logger.info("Starting feature engineering")

# ============================================================================
# FEATURE ENGINEERING FOR SELECTED FEATURES
# ============================================================================

# Handle missing review texts
missing_count = df['review_text'].isnull().sum()
if missing_count > 0:
    logger.warning(f"Found {missing_count} missing review texts, filling with empty string")
    df['review_text'] = df['review_text'].fillna('')

# 1. TEXT FEATURES: word_count
logger.info("Computing word_count from review_text")
df['word_count'] = df['review_text'].apply(lambda x: len(str(x).split()))

# 2. TEXT FEATURES: sentiment_polarity
logger.info("Computing sentiment_polarity using TextBlob")
def get_sentiment_polarity(text):
    """Extract sentiment polarity from text using TextBlob."""
    try:
        blob = TextBlob(str(text))
        return blob.sentiment.polarity
    except Exception as e:
        logger.debug(f"Sentiment extraction failed: {e}")
        return 0.0

df['sentiment_polarity'] = df['review_text'].apply(get_sentiment_polarity)

# 3. PLAYTIME FEATURES: log_playtime_at_review
logger.info("Computing log_playtime_at_review")
df['log_playtime_at_review'] = np.log1p(df['playtime_at_review_hours'])

# 4. Handle game metadata (if present)
if 'game_price' in df.columns:
    df['game_price'] = df['game_price'].fillna(0)
    logger.info("Filled missing game_price with 0")

if 'game_genre' in df.columns:
    df['game_genre'] = df['game_genre'].fillna('Unknown')
    logger.info("Filled missing game_genre with 'Unknown'")
    
if 'game_description' in df.columns:
    df['game_description'] = df['game_description'].fillna('')
    logger.info("Filled missing game_description with empty string")

# ============================================================================
# CREATE FEATURE SUBSET
# ============================================================================

features_keep = [
    # Target
    'recommended',
    
    # User history
    'user_games_owned',
    'user_num_reviews',
    'log_playtime_at_review',
    
    # GAME ENTITY (what the game IS)
    'game_price',           # Numeric
    'game_genre',           # Categorical
    'game_description',     # Free text (ENTITY ATTRIBUTE)
    
    # USER OPINION (what users THINK) - for prediction
    'review_text',          # Raw text
    'sentiment_polarity',   # Extracted from review_text
    'word_count',           # Extracted from review_text
]

# Verify all features exist
missing_features = [f for f in features_keep if f not in df.columns]
if missing_features:
    logger.error(f"Missing features in dataframe: {missing_features}")
    raise KeyError(f"Features not found: {missing_features}")

# Create subset dataframe
df_subset = df[features_keep].copy()

logger.info(f"Created feature subset with {len(features_keep)} features")
logger.info(f"Subset shape: {df_subset.shape}")

# Display summary
print("\n" + "="*60)
print("FEATURE ENGINEERING COMPLETE")
print("="*60)
print(f"Dataset shape: {df_subset.shape}")
print(f"\nSelected features:")
for i, feat in enumerate(features_keep, 1):
    print(f"  {i:2d}. {feat}")
print("="*60)

# Show preview
print("\nDataframe preview:")
df_subset.head()

INFO     | Starting feature engineering
INFO     | Computing word_count from review_text
INFO     | Computing sentiment_polarity using TextBlob
INFO     | Computing log_playtime_at_review
INFO     | Filled missing game_price with 0
INFO     | Filled missing game_genre with 'Unknown'
INFO     | Filled missing game_description with empty string
INFO     | Created feature subset with 10 features
INFO     | Subset shape: (50000, 10)

FEATURE ENGINEERING COMPLETE
Dataset shape: (50000, 10)

Selected features:
   1. recommended
   2. user_games_owned
   3. user_num_reviews
   4. log_playtime_at_review
   5. game_price
   6. game_genre
   7. game_description
   8. review_text
   9. sentiment_polarity
  10. word_count

Dataframe preview:


,recommended,user_games_owned,user_num_reviews,log_playtime_at_review,game_price,game_genre,game_description,review_text,sentiment_polarity,word_count
0,True,0,11,4.398351,0,"Action,Adventure,Massively Multiplayer,Free To...","PUBG: BATTLEGROUNDS, the high-stakes winner-ta...","The game play is solid, I started when it was ...",0.103846,75
1,True,78,2,5.865240,0,"Action,Adventure,Massively Multiplayer,Free To...","PUBG: BATTLEGROUNDS, the high-stakes winner-ta...",funny,0.250000,1
2,False,280,26,2.701361,0,"Action,Adventure,Massively Multiplayer,Free To...","PUBG: BATTLEGROUNDS, the high-stakes winner-ta...",shi8,0.000000,1
3,False,189,13,5.131180,0,"Action,Adventure,Massively Multiplayer,Free To...","PUBG: BATTLEGROUNDS, the high-stakes winner-ta...",Litteraly WHO plays this game anymore,-0.400000,6
4,True,61,6,5.389452,0,"Action,Adventure,Massively Multiplayer,Free To...","PUBG: BATTLEGROUNDS, the high-stakes winner-ta...",pog,0.000000,1


---

## 3. Model Training

Train the binary classifier to predict whether a user will recommend a game.

In [ ]:
def train_classifier(X_train, y_train, **kwargs):
    """
    Train binary classifier for game recommendation prediction.
    
    Parameters
    ----------
    X_train : array-like
        Training features
    y_train : array-like
        Training labels
    **kwargs : dict
        Additional model parameters
        
    Returns
    -------
    model
        Trained classifier
    """
    # TODO: Implement training logic
    # - Choose classifier (LogisticRegression, RandomForest, XGBoost, etc.)
    # - Handle class imbalance if needed
    # - Fit model
    # - Return trained model
    
    pass

In [ ]:
# TODO: Train the model
# model = train_classifier(X_train, y_train)

pass

---

## 4. Prediction Function (with Counterfactuals)

Create a function that:
- Takes a user + game input
- Outputs binary prediction (yes/no they'll like it)
- If prediction is negative: generates counterfactual explanations

In [ ]:
def predict_with_counterfactuals(model, user_features, threshold=0.5):
    """
    Predict game recommendation and generate counterfactuals if negative.
    
    Parameters
    ----------
    model : classifier
        Trained binary classifier
    user_features : dict or array-like
        User and game features
    threshold : float
        Classification threshold
        
    Returns
    -------
    dict
        Dictionary containing:
        - 'prediction': bool (True if recommended)
        - 'probability': float (model confidence)
        - 'counterfactuals': list (if prediction is negative)
    """
    # TODO: Implement prediction logic
    # - Get prediction probability
    # - Make binary prediction
    # - If negative: generate counterfactual explanations
    #   ("If you played X more hours, prediction would be positive")
    #   ("If review text contained Y sentiment, prediction would be positive")
    
    pass

In [ ]:
def generate_counterfactuals(model, original_features, feature_names):
    """
    Generate counterfactual explanations for negative predictions.
    
    Parameters
    ----------
    model : classifier
        Trained model
    original_features : array-like
        Original feature values
    feature_names : list
        Names of features
        
    Returns
    -------
    list
        List of counterfactual explanations
    """
    # TODO: Implement counterfactual generation
    # - Try modifying each feature
    # - Find minimal changes that flip prediction
    # - Return human-readable explanations
    
    pass

In [ ]:
# TODO: Demo prediction function
# - Create sample user input
# - Get prediction + counterfactuals
# - Display results

pass

---

## 5. Model Evaluation

Assess model performance using appropriate metrics.

In [ ]:
# TODO: Evaluate model on test set
# - Accuracy
# - Precision, Recall, F1-Score
# - ROC-AUC
# - Confusion Matrix

pass

In [ ]:
# TODO: Visualize results
# - Confusion matrix heatmap
# - ROC curve
# - Feature importance plot

pass

In [ ]:
# TODO: Print final evaluation summary

pass

---

## Summary

This notebook provides a complete pipeline for interpretable game recommendation predictions:

1. **Data Loading & Exploration** - Loaded and analyzed Steam review data
2. **Feature Engineering** - Extracted features from text and numeric data
3. **Model Training** - Trained binary classifier
4. **Prediction Function** - Generated predictions with counterfactual explanations
5. **Model Evaluation** - Assessed performance metrics

---

**Next Steps:**
- Implement remaining sections
- Tune model hyperparameters
- Improve counterfactual generation logic
- Add visualizations